<a href="https://colab.research.google.com/github/i-anshumanbaghmare/TB-Detection_ResNet50_Transfer-Learning/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TB Detection using ResNet50 Transfer-Learning

## Phase 1 : Data Loading and Preprocessing


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nuttawatsawang/chest-x-ray-dataset-montgomery-and-shenzhen")

print("Path to dataset files:", path)

100%|██████████| 4.08G/4.08G [00:39<00:00, 111MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nuttawatsawang/chest-x-ray-dataset-montgomery-and-shenzhen/versions/1


### Data Organization and Initial Setup


Organize raw image data into train/test splits for Normal and TB categories.


In [2]:
import os
import shutil
import random


# CONFIGURATION
# ============================

base_path = path
random_seed = 42
val_ratio = 0.20   # 20% of final train goes to validation (copied)
test_ratio = 0.20  # 20% of original train goes to test (moved)

random.seed(random_seed)

# Paths to raw datasets
montgomery_cxr_png_path = os.path.join(base_path, "Montgomery_Set", "Montgomery_Set", "CXR_png")
shenzhen_cxr_png_path = os.path.join(base_path, "Shenzhen_Set", "Shenzhen_Set", "CXR_png")

# Final organized paths
data_dir = os.path.join(base_path, "data")

train_normal = os.path.join(data_dir, "train", "Normal")
train_tb     = os.path.join(data_dir, "train", "TB")
val_normal   = os.path.join(data_dir, "val", "Normal")
val_tb       = os.path.join(data_dir, "val", "TB")
test_normal  = os.path.join(data_dir, "test", "Normal")
test_tb      = os.path.join(data_dir, "test", "TB")

# Create all directories
for p in [train_normal, train_tb, val_normal, val_tb, test_normal, test_tb]:
    os.makedirs(p, exist_ok=True)

print("Directory structure created.\n")


# Step 1: Collect ALL images into train/ first (using MOVE)
# ============================

print("Moving images from Montgomery and Shenzhen into train/...")

for filename in os.listdir(montgomery_cxr_png_path):
    src = os.path.join(montgomery_cxr_png_path, filename)
    if filename.endswith("_0.png"):
        shutil.move(src, os.path.join(train_normal, filename))
    elif filename.endswith("_1.png"):
        shutil.move(src, os.path.join(train_tb, filename))

for filename in os.listdir(shenzhen_cxr_png_path):
    src = os.path.join(shenzhen_cxr_png_path, filename)
    if filename.endswith("_0.png"):
        shutil.move(src, os.path.join(train_normal, filename))
    elif filename.endswith("_1.png"):
        shutil.move(src, os.path.join(train_tb, filename))

print(f"Finished collecting. Total in train:")
print(f"   Normal: {len(os.listdir(train_normal))}")
print(f"   TB:     {len(os.listdir(train_tb))}\n")


# Step 2: Create TEST set (20% — MOVE)
# ============================

def move_to_test(src_folder, dst_folder, ratio=test_ratio):
    files = os.listdir(src_folder)
    random.shuffle(files)
    n_move = int(len(files) * ratio)
    moved = 0
    for f in files[:n_move]:
        shutil.move(os.path.join(src_folder, f), os.path.join(dst_folder, f))
        moved += 1
    print(f"   Moved {moved} images from {os.path.basename(src_folder)} → test")

print("Creating test set (20% moved):")
move_to_test(train_normal, test_normal)
move_to_test(train_tb, test_tb)

print(f"After test split:")
print(f"   train/Normal: {len(os.listdir(train_normal))}")
print(f"   train/TB:     {len(os.listdir(train_tb))}")
print(f"   test/Normal:  {len(os.listdir(test_normal))}")
print(f"   test/TB:      {len(os.listdir(test_tb))}\n")


# Step 3: Create VALIDATION set (20% of REMAINING train — COPY)
# ============================

def copy_to_val(src_folder, dst_folder, ratio=val_ratio):
    files = os.listdir(src_folder)
    random.shuffle(files)
    n_copy = int(len(files) * ratio)
    copied = 0
    for f in files[:n_copy]:
        shutil.copy2(os.path.join(src_folder, f), os.path.join(dst_folder, f))
        copied += 1
    print(f"   Copied {copied} images from {os.path.basename(src_folder)} → val")

print("Creating validation set (20% of current train — copied):")
copy_to_val(train_normal, val_normal)
copy_to_val(train_tb, val_tb)


# Summary
# ============================

print("\n" + "="*50)
print("FINAL DATASET SPLIT")
print("="*50)
print(f"train/Normal : {len(os.listdir(train_normal))}")
print(f"train/TB     : {len(os.listdir(train_tb))}")
print(f"val/Normal   : {len(os.listdir(val_normal))}")
print(f"val/TB       : {len(os.listdir(val_tb))}")
print(f"test/Normal  : {len(os.listdir(test_normal))}")
print(f"test/TB      : {len(os.listdir(test_tb))}")
print("="*50)
print("Done! You can now use this structure with any PyTorch/TensorFlow trainer.")
print("   (train + val for training, test as held-out evaluation)")

Directory structure created.

Moving images from Montgomery and Shenzhen into train/...
Finished collecting. Total in train:
   Normal: 406
   TB:     394

Creating test set (20% moved):
   Moved 81 images from Normal → test
   Moved 78 images from TB → test
After test split:
   train/Normal: 325
   train/TB:     316
   test/Normal:  81
   test/TB:      78

Creating validation set (20% of current train — copied):
   Copied 65 images from Normal → val
   Copied 63 images from TB → val

FINAL DATASET SPLIT
train/Normal : 325
train/TB     : 316
val/Normal   : 65
val/TB       : 63
test/Normal  : 81
test/TB      : 78
Done! You can now use this structure with any PyTorch/TensorFlow trainer.
   (train + val for training, test as held-out evaluation)


## TensorFlow Data Pipeline Setup

Load the images from the newly created folders, apply the necessary adaptations for ResNet50V2, and set up batching.

In [4]:
import os
import tensorflow as tf
import numpy as np



print(f"Final Train TB: {len(os.listdir(train_tb))}, Final Train Normal: {len(os.listdir(train_normal))}")

Final Train TB: 316, Final Train Normal: 325
